# Exercise 1.2 – Counting Cars Towards Downtown (Traffic_Laramie_1.mp4 & Traffic_Laramie_2.mp4)

This notebook extends Exercise 1.1 by reusing the same motion-based detection and centroid tracking
(frame differencing + background subtraction) and adds a counting mechanism.

Cars are counted when their tracked centroid crosses a virtual counting line in the direction of “downtown”.
The output is:
- Total number of cars (per video, for a chosen time interval)
- Cars per minute (per video)

In [ ]:
!pip -q install opencv-python numpy matplotlib #starting form here

import cv2
import numpy as np
import matplotlib.pyplot as plt

# ---- Safe display function (works in Colab AND normal Jupyter) ----
try:
    from google.colab.patches import cv2_imshow
    def show_image(img):
        cv2_imshow(img)
except ImportError:
    def show_image(img):
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img_rgb)
        plt.axis("off")
        plt.show()

## Reusing Exercise 1.1 motion detection and tracking

As required in the coursework, motion segmentation is obtained using:
- Background subtraction (MOG2)
- Frame differencing

The two masks are combined and cleaned using morphological operations.
Contours detected in the motion mask represent candidate moving objects.
Very small contours are filtered using an area threshold because they are
typically caused by noise, shadows, or minor pixel fluctuations rather
than actual vehicles. Only contours above the threshold are treated as
valid vehicle detections.
A centroid-based tracker assigns persistent IDs across frames.

### Motion Detection and Counting Implementation

The following function processes the traffic video and performs vehicle detection, tracking, and counting.

Moving vehicles are detected using a combination of frame differencing and background subtraction. Frame differencing highlights changes between consecutive frames, while the background subtraction model isolates moving objects from the static background.

Detected vehicle regions are extracted using contour detection. Each vehicle is then tracked across frames using a centroid-based tracking approach that assigns a unique ID to each detected object.

A counting line is placed across the road leading towards the downtown area. When the centroid of a tracked vehicle crosses this line in the correct direction, the vehicle count is incremented. The tracking IDs ensure that each vehicle is counted only once.

In [ ]:
def count_cars_towards_downtown(
    video_path,
    start_sec=0,
    end_sec=120,              # choose interval here
    line_pos_ratio=0.65,      # position of counting line inside ROI (0..1 from top of ROI)
    direction="left_to_right",# or "right_to_left"
    show_preview_every=30     # show every N frames, None to disable
):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Jump to start time
    start_frame = int(start_sec * fps)
    end_frame = int(end_sec * fps) if end_sec is not None else total_frames
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # Read first frame to define ROI
    ret, frame0 = cap.read()
    if not ret:
        raise RuntimeError("Could not read first frame at start time.")

    h, w = frame0.shape[:2]
    x1, x2 = int(w * 0.45), w
    y1, y2 = 0, int(h * 0.75)

    roi_h = (y2 - y1)
    roi_w = (x2 - x1)

    # Downtown "zone" polygon (in ROI coordinates).
    # Any car track entering this zone will be counted once.
    downtown_zone = np.array([
        [int(roi_w * 0.65), int(roi_h * 0.10)],
        [int(roi_w * 0.98), int(roi_h * 0.10)],
        [int(roi_w * 0.98), int(roi_h * 0.55)],
        [int(roi_w * 0.65), int(roi_h * 0.55)],
    ], dtype=np.int32)

    # Motion components (same as 1.1)
    bg_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
    prev_gray = None

    # Tracker (same logic as 1.1)
    next_id = 1
    tracks = {}         # id -> (cx, cy)
    missed = {}         # id -> missed count
    max_match_dist = 60
    max_missed_frames = 20

    # For counting
    counted_ids = set()
    last_positions = {}  # id -> previous (cx, cy)

    # Helper
    def box_centroid(x, y, ww, hh):
        return (int(x + ww/2), int(y + hh/2))

    # Warm-up frames for background model
    warmup = 60

    frame_idx = start_frame
    while True:
        if frame_idx > end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        roi = frame[y1:y2, x1:x2]

        # Background subtraction + shadow removal
        fg = bg_sub.apply(roi)
        _, fg_mask = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

        # Frame differencing
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        if prev_gray is None:
            prev_gray = gray
            continue

        diff = cv2.absdiff(gray, prev_gray)
        _, diff_mask = cv2.threshold(diff, 15, 255, cv2.THRESH_BINARY)
        prev_gray = gray

        # allow background model to stabilise
        if frame_idx < start_frame + warmup:
            continue

        # Combine + morphology
        motion_mask = cv2.bitwise_or(fg_mask, diff_mask)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
        motion_mask = cv2.dilate(motion_mask, kernel, iterations=2)

        # Detect
        contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        detections = []
        for c in contours:
            area = cv2.contourArea(c)
            if area < 900:
                continue
            x, y, ww, hh = cv2.boundingRect(c)
            cx, cy = box_centroid(x, y, ww, hh)
            detections.append({"box": (x, y, ww, hh), "c": (cx, cy), "id": None})

        # Increase missed counts
        for tid in list(missed.keys()):
            missed[tid] += 1

        # Match detections to existing tracks (greedy nearest neighbour)
        pairs = []
        for di, det in enumerate(detections):
            cx, cy = det["c"]
            for tid, (tx, ty) in tracks.items():
                d = ((cx - tx)**2 + (cy - ty)**2) ** 0.5
                if d <= max_match_dist:
                    pairs.append((d, tid, di))

        pairs.sort(key=lambda x: x[0])
        used_tracks, used_dets = set(), set()

        for d, tid, di in pairs:
            if tid in used_tracks or di in used_dets:
                continue
            used_tracks.add(tid)
            used_dets.add(di)
            detections[di]["id"] = tid
            tracks[tid] = detections[di]["c"]
            missed[tid] = 0

        # New IDs
        for det in detections:
            if det["id"] is None:
                det["id"] = next_id
                tracks[next_id] = det["c"]
                missed[next_id] = 0
                next_id += 1

        # Delete old tracks
        for tid in list(missed.keys()):
            if missed[tid] > max_missed_frames:
                missed.pop(tid, None)
                tracks.pop(tid, None)
                last_positions.pop(tid, None)

        # ---- COUNTING LOGIC (COMPLIANT: FROM ALL DIRECTIONS) ----
        # Count a car ONCE when its tracked centroid moves from OUTSIDE -> INSIDE the downtown zone.
        for det in detections:
            tid = det["id"]
            cx, cy = det["c"]

            prev = last_positions.get(tid, None)
            last_positions[tid] = (cx, cy)

            if prev is None:
                continue

            prev_inside = cv2.pointPolygonTest(downtown_zone, prev, False) >= 0
            curr_inside = cv2.pointPolygonTest(downtown_zone, (cx, cy), False) >= 0

            # Count only on entry (outside -> inside)
            if (not prev_inside) and curr_inside and tid not in counted_ids:
                counted_ids.add(tid)

        # Preview
        if show_preview_every is not None and frame_idx % show_preview_every == 0:
            vis = roi.copy()

            # draw downtown zone
            cv2.polylines(vis, [downtown_zone], isClosed=True, color=(0, 0, 255), thickness=2)

            cv2.putText(vis, f"Count={len(counted_ids)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

            for det in detections:
                x, y, ww, hh = det["box"]
                tid = det["id"]
                cx, cy = det["c"]

                cv2.rectangle(vis, (x, y), (x + ww, y + hh), (0, 255, 0), 2)
                cv2.putText(vis, f"ID {tid}", (x, max(20, y - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                cv2.circle(vis, (cx, cy), 3, (255, 255, 0), -1)

            print(f"Frame {frame_idx} | Counted={len(counted_ids)} | Active tracks={len(tracks)}")
            show_image(vis)

    cap.release()

    duration_sec = (end_sec - start_sec) if end_sec is not None else (total_frames / fps - start_sec)
    cars_per_min = len(counted_ids) / (duration_sec / 60.0)

    return {
        "video": video_path,
        "start_sec": start_sec,
        "end_sec": end_sec,
        "total_cars": len(counted_ids),
        "cars_per_min": cars_per_min
    }


## Counting cars towards downtown

A **vertical** counting line is placed inside the main-street ROI.
Each vehicle is tracked with a unique ID (centroid tracker).

A car is counted **once** when its tracked centroid crosses the counting line in the
chosen “downtown” direction (**left_to_right** in this notebook). To prevent double
counting, the application stores IDs that have already crossed the line. To satisfy ‘from all directions’, the algorithm counts a vehicle when its tracked centroid enters a defined ‘downtown zone’ polygon, rather than relying on a single line crossing.

Vehicles approaching the downtown area from different lanes may
cross the counting line from multiple directions. The centroid-based
tracking ensures that each vehicle is counted only once regardless
of the path taken.

In [ ]:
# Choose interval (example: first 2 minutes)
start_sec = 0
end_sec = 120

# Run on both videos
# Note: line_pos_ratio and direction are kept for compatibility with older line-based version,
# but this solution uses a downtown polygon zone (counts from all directions).
res1 = count_cars_towards_downtown(
    "Traffic_Laramie_1.mp4",
    start_sec=start_sec, end_sec=end_sec,
    line_pos_ratio=0.65,
    direction="left_to_right",
    show_preview_every=30
)

res2 = count_cars_towards_downtown(
    "Traffic_Laramie_2.mp4",
    start_sec=start_sec, end_sec=end_sec,
    line_pos_ratio=0.65,
    direction= "left_to_right",
    show_preview_every=30
)

print(res1)
print(res2)

## Results table (Task 1.2)

The table below reports the total number of cars moving towards downtown and the
average cars-per-minute for the selected time interval.

In [ ]:
def format_table(res1, res2):
    print("| Video | Total number of cars | Cars per minute |")
    print("|---|---:|---:|")
    print(f"| {res1['video']} | {res1['total_cars']} | {res1['cars_per_min']:.2f} |")
    print(f"| {res2['video']} | {res2['total_cars']} | {res2['cars_per_min']:.2f} |")

format_table(res1, res2)